In [1]:
import pandas as pd

# Load RNA-RNA interaction file
file_path = "data/RNAInter_RNA_RNA.txt"

# Try common delimiters (RNAInter files are usually tab-delimited)
df = pd.read_csv(file_path, sep='\t', low_memory=False)

# Display basic info
print(f"Number of interactions: {len(df)}")
df.head()


Number of interactions: 9484609


,RNAInterID,Interactor1.Symbol,Category1,Species1,Interactor2.Symbol,Category2,Species2,Raw_ID1,Raw_ID2,score,strong,weak,predict
0,RR00000169,(Ca)N,repeats,Mus musculus,B1_Mus1,repeats,Mus musculus,chr4(CA)n125049227,chr15B1_Mus136600585,0.23,NaN,MARIO,NaN
1,RR00000238,(Ca)N,repeats,Mus musculus,B1_Mus1,repeats,Mus musculus,chr8(CA)n67899847,chr8B1_Mus167899573,0.23,NaN,MARIO,NaN
2,RR00000198,(Ca)N,repeats,Mus musculus,B1_Mus1,repeats,Mus musculus,chr6(CA)n135011645,chr6B1_Mus1135012241,0.23,NaN,MARIO,NaN
3,RR00000231,(Ca)N,repeats,Mus musculus,B1_Mus1,repeats,Mus musculus,chr8(CA)n114753246,chr8B1_Mus1114753067,0.23,NaN,MARIO,NaN
4,RR00000249,(Ca)N,repeats,Mus musculus,B3,repeats,Mus musculus,chr9(CA)n21755504,chr9B321755686,0.23,NaN,MARIO,NaN


In [2]:
# Show column names
print("Columns:", df.columns.tolist())

# Inspect data types
df.dtypes


Columns: ['RNAInterID', 'Interactor1.Symbol', 'Category1', 'Species1', 'Interactor2.Symbol', 'Category2', 'Species2', 'Raw_ID1', 'Raw_ID2', 'score', 'strong', 'weak', 'predict']


RNAInterID             object
Interactor1.Symbol     object
Category1              object
Species1               object
Interactor2.Symbol     object
Category2              object
Species2               object
Raw_ID1                object
Raw_ID2                object
score                 float64
strong                 object
weak                   object
predict                object
dtype: object

In [3]:
# Check unique categories in both interactors
cat1_unique = df['Category1'].unique()
cat2_unique = df['Category2'].unique()

# Optionally, combined unique categories across both columns
combined_categories = set(cat1_unique) | set(cat2_unique)
print("All unique RNA categories in dataset:", combined_categories)



All unique RNA categories in dataset: {'tRNA', 'miscRNA', 'lincRNA', 'unassigned RNA', 'Protein', 'shRNA', 'antisense', 'processed_transcript', 'repeats', 'others', 'sncRNA', 'pseudo', 'sRNA', 'ncRNA', 'TEC', 'rRNA', 'eRNA', 'ribozyme', 'tRF', 'scaRNA', 'circRNA', 'unknown', 'PCG', 'mRNA', 'Category1', 'miRNA', 'Category2', 'lncRNA', 'protein', 'TR_C_gene', 'piRNA', 'snoRNA', 'scRNA', 'TF', 'snRNA'}


In [4]:
exclude_categories = {'Category1', 'Category2', 'unknown', 'unassigned RNA', 'others', 'Protein', 'protein',
                      'PCG', # Protein-coding Gene
                      'TF', # transcription factor gene
                      'TR_C_gene', # T-cell receptor constant region gene; immune-related.
                      'antisense', # transcripts complementary to sense strands of annotated genes.
                      'TEC', # “to be experimentally confirmed,” provisional transcripts.
                      'repeats' # transcripts overlapping repetitive genomic elements.
                     }

# Filter out unwanted categories from both interactors
df_filt = df[~df['Category1'].isin(exclude_categories) & ~df['Category2'].isin(exclude_categories)]

print(f"Rows after removing not-clear categories: {len(df_filt)}")


Rows after removing not-clear categories: 9429474


In [5]:
# Check unique categories in both interactors
cat1_unique = df_filt['Category1'].unique()
cat2_unique = df_filt['Category2'].unique()

# Optionally, combined unique categories across both columns
combined_categories = set(cat1_unique) | set(cat2_unique)
print("All unique RNA categories in dataset:", combined_categories)

All unique RNA categories in dataset: {'tRNA', 'miscRNA', 'lincRNA', 'shRNA', 'processed_transcript', 'sncRNA', 'pseudo', 'sRNA', 'ncRNA', 'rRNA', 'eRNA', 'ribozyme', 'tRF', 'scaRNA', 'circRNA', 'mRNA', 'miRNA', 'lncRNA', 'piRNA', 'snoRNA', 'scRNA', 'snRNA'}


In [6]:
# initial count
print(f"RNA–RNA interactions: {len(df_filt)}")

# keep rows where each interactor has at least one identifier
mask_valid_1 = df_filt[['Raw_ID1', 'Interactor1.Symbol']].notna().any(axis=1)
mask_valid_2 = df_filt[['Raw_ID2', 'Interactor2.Symbol']].notna().any(axis=1)

df_filt = df_filt[mask_valid_1 & mask_valid_2]

# final count
print(f"RNA–RNA interactions: {len(df_filt)}")


RNA–RNA interactions: 9429474
RNA–RNA interactions: 9429405


In [7]:
parsed = []

def parse_ids(id_value):
    if pd.isna(id_value):
        return [None]
    elif ':' in id_value:
        prefix, ids_str = id_value.split(':', 1)
        return [f"{prefix}:{id_.strip()}" for id_ in ids_str.split(';')]
    else:
        parsed.append(id_value)
        return [id_value.strip()]

# Add list columns
df_filt['Raw_ID1_list'] = df_filt['Raw_ID1'].apply(parse_ids)
df_filt['Raw_ID2_list'] = df_filt['Raw_ID2'].apply(parse_ids)


In [8]:
# we need to standardise the species to TAXIDs
# 1. See unique species
df_filt.replace({'-': None}, inplace=True)
unique_species = pd.unique(df_filt[['Species1', 'Species2']].values.ravel())
print("Unique species:", len(unique_species))

from ete3 import NCBITaxa
ncbi = NCBITaxa()

unique_species_taxids = {}
missing = []

for test_species in unique_species:
    # Skip placeholders
    if test_species in [None]:
        continue

    # Handle known special cases first
    if test_species == 'Solanum tuberosum L.':
        unique_species_taxids[test_species] = "4113"
        continue
    if test_species == 'Rhodobacter sphaeroides ATCC 17025':
        unique_species_taxids[test_species] = "1061"
        continue

    if test_species == 'Luffa cylindrica':
        unique_species_taxids[test_species] = "3669"
        continue

    # Query NCBI
    try:
        taxid = str(ncbi.get_name_translator([test_species])[test_species][0])
        unique_species_taxids[test_species] = taxid
    except KeyError:
        missing.append(test_species)

print("Mapped species (unique -1 for None):", len(unique_species_taxids))
print("Missing species:", missing)


Unique species: 135
Mapped species (unique -1 for None): 134
Missing species: []


In [9]:
# Assume df is your original dataframe
# unique_species_taxids: dict {species_name -> taxid}

df_filt['Species1_TaxID'] = df_filt['Species1'].map(unique_species_taxids)
df_filt['Species2_TaxID'] = df_filt['Species2'].map(unique_species_taxids)

unmapped_s1 = df_filt.loc[df_filt['Species1'].notna() & df_filt['Species1_TaxID'].isna(), 'Species1'].unique()
unmapped_s2 = df_filt.loc[df_filt['Species2'].notna() & df_filt['Species2_TaxID'].isna(), 'Species2'].unique()

print("Unmapped Species1:", unmapped_s1)
print("Unmapped Species2:", unmapped_s2)

Unmapped Species1: []
Unmapped Species2: []


In [10]:
print(f"RNA–RNA interactions: {len(df_filt)}")
# removing NAN
df_filt = df_filt.dropna(subset=['Species1_TaxID', 'Species2_TaxID'])
print(f"RNA–RNA interactions: {len(df_filt)}")

RNA–RNA interactions: 9429405
RNA–RNA interactions: 9429404


In [11]:
# Check unique categories in both interactors
cat1_unique = df_filt['Category1'].unique()
cat2_unique = df_filt['Category2'].unique()

# Optionally, combined unique categories across both columns
combined_categories = set(cat1_unique) | set(cat2_unique)
print("All unique RNA categories in dataset:", combined_categories)

RNA_TYPE_MAP = {
    'processed_transcript': 'ncRNA',
    'miscRNA': 'ncRNA',
    'misc_RNA': 'ncRNA',
}

# Apply standardisation
df_filt['Category1_std'] = df_filt['Category1'].replace(RNA_TYPE_MAP)
df_filt['Category2_std'] = df_filt['Category2'].replace(RNA_TYPE_MAP)

# Recompute unique categories after mapping
cat1_std = set(df_filt['Category1_std'].unique())
cat2_std = set(df_filt['Category2_std'].unique())

print("Standardised categories:", cat1_std | cat2_std)

assert not df_filt[['Category1_std', 'Category2_std']].isna().any().any()

All unique RNA categories in dataset: {'tRNA', 'miscRNA', 'lincRNA', 'shRNA', 'processed_transcript', 'sncRNA', 'pseudo', 'sRNA', 'ncRNA', 'rRNA', 'eRNA', 'ribozyme', 'tRF', 'scaRNA', 'circRNA', 'mRNA', 'miRNA', 'lncRNA', 'piRNA', 'snoRNA', 'scRNA', 'snRNA'}
Standardised categories: {'tRNA', 'rRNA', 'eRNA', 'ribozyme', 'tRF', 'scaRNA', 'lincRNA', 'circRNA', 'mRNA', 'miRNA', 'shRNA', 'lncRNA', 'piRNA', 'snoRNA', 'sncRNA', 'pseudo', 'sRNA', 'scRNA', 'ncRNA', 'snRNA'}


In [12]:
# DATASET IS READY.

# Now we need to extract sequences, we start by taking unique of ('Interactor1.Symbol', 'Category1', 'Species1_TaxID', 'Raw_ID1_list')
# We do not care about 1 and 2, we can create the final database later.

def normalize_id_list(x):
    if isinstance(x, list):
        return tuple(sorted(str(i) for i in x if i is not None))
    return tuple()
df_filtbackup = df_filt

df_filt = df_filt.copy()
df_filt['Raw_ID1_key'] = df_filt['Raw_ID1_list'].apply(normalize_id_list)
df_filt['Raw_ID2_key'] = df_filt['Raw_ID2_list'].apply(normalize_id_list)

rna_entities_1 = (
    df_filt.loc[:, ['Interactor1.Symbol', 'Category1', 'Species1_TaxID', 'Raw_ID1_key']]
    .drop_duplicates()
    .rename(columns={
        'Interactor1.Symbol': 'Symbol',
        'Category1': 'Category',
        'Species1_TaxID': 'Species',
        'Raw_ID1_key': 'Raw_ID_key'
    })
)

rna_entities_2 = (
    df_filt.loc[:, ['Interactor2.Symbol', 'Category2', 'Species2_TaxID', 'Raw_ID2_key']]
    .drop_duplicates()
    .rename(columns={
        'Interactor2.Symbol': 'Symbol',
        'Category2': 'Category',
        'Species2_TaxID': 'Species',
        'Raw_ID2_key': 'Raw_ID_key'
    })
)

rna_entities = (
    pd.concat([rna_entities_1, rna_entities_2], ignore_index=True)
    .drop_duplicates()
)

print(f"Unique RNA entities: {len(rna_entities)}")

rna_entities['RNA_IDs'] = rna_entities['Raw_ID_key'].apply(list)

def classify_ids(id_list):
    return {
        'miRBase': [],
        'NCBI': [],
        'Ensembl': [],
        'circBase': [],
        'tRFdb': [],
        'GenBank': [],
        'Other': [],
        'RNAcentral': []   # ← to be filled after resolution
    }

def populate_buckets(id_list):
    buckets = classify_ids(id_list)
    for rid in id_list:
        if rid.startswith('miRBase:'):
            buckets['miRBase'].append(rid.split(':', 1)[1])
        elif rid.startswith('NCBI:'):
            buckets['NCBI'].append(rid.split(':', 1)[1])
        elif rid.startswith('Ensembl:'):
            buckets['Ensembl'].append(rid.split(':', 1)[1])
        elif rid.startswith('circBase:'):
            buckets['circBase'].append(rid.split(':', 1)[1])
        elif rid.startswith('tRFdb:'):
            buckets['tRFdb'].append(rid.split(':', 1)[1])
        elif rid.startswith('GenBank:'):
            buckets['GenBank'].append(rid.split(':', 1)[1])
        else:
            buckets['Other'].append(rid)
    return buckets

rna_entities['ID_buckets'] = rna_entities['RNA_IDs'].apply(populate_buckets)


Unique RNA entities: 290684


In [13]:
# now we remove all mRNA entities
# Remove mRNA entities
print(len(rna_entities))
m_rna_entities = rna_entities[rna_entities["Category"].str.lower() == "mrna"].copy()
m_rna_entities.to_pickle("data/m_rna_entities_to_be_processed.pkl")
print(f"Unique mRNA entities: {len(m_rna_entities)}")
ncrna_entities = rna_entities[rna_entities["Category"].str.lower() != "mrna"].copy()
ncrna_entities.to_pickle("data/nc_rna_entities_to_be_processed.pkl")
print(f"Unique RNA entities (excluding mRNA): {len(rna_entities)}")

import pandas as pd
df_filt.to_pickle("data/interactions_processed_noRNAcentralIDs.pkl")

290684
Unique mRNA entities: 213723
Unique RNA entities (excluding mRNA): 290684


In [14]:
#--------------------------------------------------------------------------------